In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────────
!pip install ultralytics timm --quiet

import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import os, random, shutil, math
from pathlib import Path
from ultralytics import YOLO
from ultralytics.nn.modules.block import C2f
from ultralytics.models.yolo.detect import DetectionTrainer

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
# ── Dataset paths ──────────────────────────────────────────────────────────────
TRAIN_IMG_SRC = "/kaggle/input/datasets/ailienpham/taco-balanced/train/images"
TRAIN_LBL_SRC = "/kaggle/input/datasets/ailienpham/taco-balanced/train/labels"

NEW_TRAIN_IMG = "/kaggle/working/taco/train/images"
NEW_TRAIN_LBL = "/kaggle/working/taco/train/labels"
VAL_IMG       = "/kaggle/working/taco/val/images"
VAL_LBL       = "/kaggle/working/taco/val/labels"

for d in [NEW_TRAIN_IMG, NEW_TRAIN_LBL, VAL_IMG, VAL_LBL]:
    os.makedirs(d, exist_ok=True)

CLASS_NAMES = [
    'Aluminium foil', 'Bottle cap', 'Bottle', 'Broken glass', 'Can',
    'Carton', 'Cigarette', 'Cup', 'Lid', 'Other litter',
    'Other plastic', 'Paper', 'Plastic bag - wrapper',
    'Plastic container', 'Pop tab', 'Straw', 'Styrofoam piece', 'Unlabeled litter'
]
# Hard classes: Broken glass(3), Cigarette(6), Other plastic(10), Unlabeled litter(17)
HARD_CLASSES     = {3, 6, 10, 17}
OVERSAMPLE_TIMES = 3


def classes_in_label(lbl_path):
    found = set()
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                p = line.strip().split()
                if p:
                    found.add(int(p[0]))
    return found


def copy_pair(img_name, dest_img_dir, dest_lbl_dir, suffix=""):
    stem, ext = os.path.splitext(img_name)
    shutil.copy(
        os.path.join(TRAIN_IMG_SRC, img_name),
        os.path.join(dest_img_dir, stem + suffix + ext)
    )
    lbl_src = os.path.join(TRAIN_LBL_SRC, stem + ".txt")
    if os.path.exists(lbl_src):
        shutil.copy(lbl_src, os.path.join(dest_lbl_dir, stem + suffix + ".txt"))


all_images = sorted([f for f in os.listdir(TRAIN_IMG_SRC) if f.endswith(".jpg")])
random.seed(42)
random.shuffle(all_images)

split      = int(0.8 * len(all_images))
train_imgs = all_images[:split]
val_imgs   = all_images[split:]

# Copy validation images
for img in val_imgs:
    stem = os.path.splitext(img)[0]
    shutil.copy(os.path.join(TRAIN_IMG_SRC, img), os.path.join(VAL_IMG, img))
    lbl = os.path.join(TRAIN_LBL_SRC, stem + ".txt")
    if os.path.exists(lbl):
        shutil.copy(lbl, os.path.join(VAL_LBL, stem + ".txt"))

# Copy + oversample training images
oversampled = 0
for img in train_imgs:
    copy_pair(img, NEW_TRAIN_IMG, NEW_TRAIN_LBL)
    stem = os.path.splitext(img)[0]
    lbl_path = os.path.join(TRAIN_LBL_SRC, stem + ".txt")
    if classes_in_label(lbl_path) & HARD_CLASSES:
        for k in range(1, OVERSAMPLE_TIMES):
            copy_pair(img, NEW_TRAIN_IMG, NEW_TRAIN_LBL, suffix=f"_aug{k}")
            oversampled += 1

print(f"Train images : {len(os.listdir(NEW_TRAIN_IMG))} (incl. {oversampled} oversampled hard-class copies)")
print(f"Val   images : {len(os.listdir(VAL_IMG))}")

In [ ]:
# ── Write taco.yaml ────────────────────────────────────────────────────────────
yaml_content = f"""path: /kaggle/working/taco
train: train/images
val:   val/images

nc: {len(CLASS_NAMES)}
names: {CLASS_NAMES}
"""

with open("/kaggle/working/taco.yaml", "w") as f:
    f.write(yaml_content)

print("taco.yaml written:")
print(yaml_content)

In [ ]:
# ── EfficientViT Cascaded Group Attention ──────────────────────────────────────

class Conv2d_BN(nn.Module):
    """Conv2d + BatchNorm, standard building block."""
    def __init__(self, in_ch, out_ch, kernel=1, stride=1, padding=0, groups=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel, stride, padding,
                              groups=groups, bias=False)
        self.bn   = nn.BatchNorm2d(out_ch)

    def forward(self, x):
        return self.bn(self.conv(x))


class CascadedGroupAttention(nn.Module):
    """
    EfficientViT Cascaded Group Attention (CGA).
    Reference: EfficientViT: Memory Efficient Vision Transformer with
    Cascaded Group Attention. Liu et al., CVPR 2023.

    Key idea:
    - Split input channels into num_heads groups
    - Each head attends to its own group, passes output to next head (cascaded)
    - Depthwise conv provides local context before attention
    - Much cheaper than full self-attention
    """
    def __init__(self, dim, num_heads=4, key_dim=16, attn_ratio=4,
                 resolution=40):
        super().__init__()
        self.num_heads = num_heads
        self.key_dim   = key_dim
        self.d         = int(attn_ratio * key_dim)
        self.scale     = key_dim ** -0.5
        self.resolution = resolution
        N = resolution * resolution

        head_dim = dim // num_heads

        # Per-head Q, K, V projections
        self.q_projs = nn.ModuleList([Conv2d_BN(head_dim, key_dim) for _ in range(num_heads)])
        self.k_projs = nn.ModuleList([Conv2d_BN(head_dim, key_dim) for _ in range(num_heads)])
        self.v_projs = nn.ModuleList([Conv2d_BN(head_dim, self.d)  for _ in range(num_heads)])

        # Depthwise conv per head for local context (kernel=5)
        self.dw_convs = nn.ModuleList([
            Conv2d_BN(head_dim, head_dim, kernel=5, padding=2, groups=head_dim)
            for _ in range(num_heads)
        ])

        # Output projection
        self.proj     = Conv2d_BN(self.d * num_heads, dim)
        self.proj_act = nn.ReLU()

        # Learnable positional bias: one bias value per key position (size N)
        # shape: (num_heads, N) — each head, each spatial position
        self.attention_biases = nn.Parameter(torch.zeros(num_heads, N))

    def forward(self, x):
        B, C, H, W = x.shape
        N = H * W
        head_dim = C // self.num_heads

        # Split channels into per-head chunks
        x_heads = x.split(head_dim, dim=1)  # list of (B, head_dim, H, W)

        attn_outputs = []
        prev = None

        for i in range(self.num_heads):
            xi = x_heads[i]

            # Local context via depthwise conv
            xi = self.dw_convs[i](xi)

            # Cascade: add previous head output if available
            if prev is not None:
                xi = xi + prev

            # Q, K, V
            q = self.q_projs[i](xi)  # (B, key_dim, H, W)
            k = self.k_projs[i](xi)
            v = self.v_projs[i](xi)  # (B, d, H, W)

            # Flatten spatial for attention
            q_flat = q.reshape(B, self.key_dim, N).permute(0, 2, 1)  # (B, N, key_dim)
            k_flat = k.reshape(B, self.key_dim, N)                    # (B, key_dim, N)
            v_flat = v.reshape(B, self.d, N).permute(0, 2, 1)        # (B, N, d)

            # Attention scores
            attn = (q_flat @ k_flat) * self.scale  # (B, N, N)

            # Positional bias: shape (N,) broadcast over queries -> (1, 1, N) -> (B, N, N)
            # Each key position gets a learned bias regardless of query position
            if N == self.resolution * self.resolution:
                pos_bias = self.attention_biases[i]          # (N,)
                attn = attn + pos_bias.unsqueeze(0).unsqueeze(0)  # (B, N, N)

            attn = attn.softmax(dim=-1)

            # Weighted sum
            out = (attn @ v_flat).permute(0, 2, 1).reshape(B, self.d, H, W)
            attn_outputs.append(out)

            # Prepare cascade input for next head: slice to head_dim channels
            prev = out[:, :head_dim, :, :]

        # Concat and project
        out = torch.cat(attn_outputs, dim=1)   # (B, d * num_heads, H, W)
        out = self.proj_act(self.proj(out))     # (B, C, H, W)
        return out


class EfficientViTBlock(nn.Module):
    """
    One EfficientViT block: FFN -> CGA -> FFN with residual connections.
    Layer scale starts tiny (1e-5) so training is stable from epoch 1.
    """
    def __init__(self, dim, num_heads=4, key_dim=16, attn_ratio=4,
                 resolution=40, expansion=4):
        super().__init__()

        self.ffn1 = nn.Sequential(
            Conv2d_BN(dim, dim * expansion, kernel=1),
            nn.GELU(),
            Conv2d_BN(dim * expansion, dim, kernel=1),
        )

        self.attn = CascadedGroupAttention(
            dim=dim, num_heads=num_heads, key_dim=key_dim,
            attn_ratio=attn_ratio, resolution=resolution,
        )

        self.ffn2 = nn.Sequential(
            Conv2d_BN(dim, dim * expansion, kernel=1),
            nn.GELU(),
            Conv2d_BN(dim * expansion, dim, kernel=1),
        )

        # Tiny init = stable early training, grows as needed
        self.ls1 = nn.Parameter(torch.ones(dim, 1, 1) * 1e-5)
        self.ls2 = nn.Parameter(torch.ones(dim, 1, 1) * 1e-5)
        self.ls3 = nn.Parameter(torch.ones(dim, 1, 1) * 1e-5)

    def forward(self, x):
        x = x + self.ls1 * self.ffn1(x)
        x = x + self.ls2 * self.attn(x)
        x = x + self.ls3 * self.ffn2(x)
        return x


print("EfficientViT CGA modules defined.")

# Sanity check
with torch.no_grad():
    dummy = torch.randn(2, 256, 40, 40)
    block = EfficientViTBlock(dim=256, num_heads=4, key_dim=32,
                              attn_ratio=4, resolution=40)
    out = block(dummy)
    assert out.shape == dummy.shape, f"Shape mismatch: {out.shape} vs {dummy.shape}"
    print(f"Sanity check passed: input {dummy.shape} -> output {out.shape}")
    params = sum(p.numel() for p in block.parameters())
    print(f"EfficientViTBlock parameters: {params:,}")


In [ ]:
# ── C2f + EfficientViT block ───────────────────────────────────────────────────

class C2f_EfficientViT(nn.Module):
    """
    Drop-in replacement for YOLOv8's C2f module.

    Structure: C2f (local features) -> EfficientViT CGA (global context)

    C2f handles local feature extraction (existing YOLO strength).
    EfficientViT CGA adds global context on top (what YOLO lacks).
    Together: local + global = better detection of deformable trash objects.
    """
    def __init__(self, c1, c2, n=1, shortcut=True,
                 num_heads=4, key_dim=32, resolution=40):
        super().__init__()

        # Original C2f block — unchanged YOLO local feature extractor
        self.c2f = C2f(c1, c2, n, shortcut)

        # EfficientViT block — global context
        # Ensure num_heads divides c2 evenly
        actual_heads = num_heads
        while c2 % actual_heads != 0 and actual_heads > 1:
            actual_heads -= 1

        self.evit = EfficientViTBlock(
            dim=c2,
            num_heads=actual_heads,
            key_dim=key_dim,
            attn_ratio=4,
            resolution=resolution,
        )

    def forward(self, x):
        # Local features from C2f
        feat = self.c2f(x)
        # Global context from EfficientViT CGA
        feat = self.evit(feat)
        return feat


print("C2f_EfficientViT defined.")

# Sanity check for layer 6 dimensions (c1=256, c2=256)
with torch.no_grad():
    dummy = torch.randn(2, 256, 40, 40)
    hybrid = C2f_EfficientViT(c1=256, c2=256, n=2,
                               num_heads=4, key_dim=32, resolution=40)
    out = hybrid(dummy)
    assert out.shape == dummy.shape, f"Shape mismatch: {out.shape}"
    print(f"C2f_EfficientViT sanity check passed: {dummy.shape} -> {out.shape}")
    total = sum(p.numel() for p in hybrid.parameters())
    print(f"C2f_EfficientViT total parameters: {total:,}")

In [ ]:
# ── Custom Trainer with EfficientViT at layer 6 ────────────────────────────────

TARGET_LAYER = 6
LAYER_N      = 2

class EfficientViTYOLOTrainer(DetectionTrainer):
    """
    YOLOv8s trainer that replaces layer 6 C2f with C2f_EfficientViT.

    Key improvements over previous version:
    1. NO backbone freezing — layer scale (1e-5 init) handles stability
    2. Differential learning rates:
       - EfficientViT block: full lr (learns fast, it's new)
       - YOLO pretrained weights: 0.1x lr (preserves pretrained knowledge)
    3. This is the correct way to fine-tune hybrid models
    """

    def get_model(self, cfg=None, weights=None, verbose=True):
        model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)

        layer = model.model[TARGET_LAYER]
        print(f"\nLayer {TARGET_LAYER} type: {type(layer).__name__}")

        if not (hasattr(layer, 'cv1') and hasattr(layer, 'cv2')):
            print(f"WARNING: Layer {TARGET_LAYER} is not C2f. Skipping.")
            return model

        c1 = layer.cv1.conv.in_channels
        c2 = layer.cv2.conv.out_channels
        n  = len(layer.m) if hasattr(layer, 'm') else LAYER_N
        print(f"Layer {TARGET_LAYER} dims: c1={c1}, c2={c2}, n={n}")

        new_layer = C2f_EfficientViT(
            c1=c1, c2=c2, n=n,
            num_heads=4,
            key_dim=32,
            resolution=40
        )

        # Copy YOLO graph metadata
        new_layer.f    = layer.f
        new_layer.i    = layer.i
        new_layer.type = layer.type
        if hasattr(layer, 'np'):
            new_layer.np = layer.np

        # Copy pretrained C2f weights
        new_layer.c2f.load_state_dict(layer.state_dict())
        print(f"Pretrained C2f weights loaded successfully")

        model.model[TARGET_LAYER] = new_layer
        print(f"Layer {TARGET_LAYER} replaced with C2f_EfficientViT")
        print(f"  C2f params      : {sum(p.numel() for p in new_layer.c2f.parameters()):,}")
        print(f"  EfficientViT    : {sum(p.numel() for p in new_layer.evit.parameters()):,}")

        return model

    def build_optimizer(self, model, name="AdamW", lr=0.001, momentum=0.937,
                        decay=0.0005, iterations=1e5):
        """
        Differential learning rates:
        - EfficientViT parameters: full lr (new weights, need to learn fast)
        - All other YOLO parameters: lr * 0.1 (pretrained, preserve knowledge)
        """
        evit_params   = []
        other_params_decay    = []
        other_params_nodecay  = []

        for name_p, param in model.named_parameters():
            if not param.requires_grad:
                continue
            if 'evit' in name_p:
                evit_params.append(param)
            elif param.ndim <= 1 or name_p.endswith('.bias'):
                other_params_nodecay.append(param)
            else:
                other_params_decay.append(param)

        param_groups = [
            # EfficientViT: full learning rate
            {'params': evit_params,
             'lr': lr,
             'weight_decay': decay,
             'name': 'efficientvit'},
            # YOLO pretrained weights with decay: 10x lower lr
            {'params': other_params_decay,
             'lr': lr * 0.1,
             'weight_decay': decay,
             'name': 'yolo_decay'},
            # YOLO pretrained weights no decay: 10x lower lr
            {'params': other_params_nodecay,
             'lr': lr * 0.1,
             'weight_decay': 0.0,
             'name': 'yolo_nodecay'},
        ]

        optimizer = torch.optim.AdamW(param_groups,
                                      lr=lr,
                                      betas=(momentum, 0.999))

        evit_count  = sum(p.numel() for p in evit_params)
        other_count = sum(p.numel() for p in other_params_decay + other_params_nodecay)
        print(f"\nDifferential LR optimizer built:")
        print(f"  EfficientViT params : {evit_count:,}  @ lr={lr}")
        print(f"  YOLO params         : {other_count:,}  @ lr={lr*0.1}")

        return optimizer


print("EfficientViTYOLOTrainer defined (differential LR, no freezing).")


In [ ]:
# ── Train hybrid model ─────────────────────────────────────────────────────────
yolo_hybrid = YOLO("yolov8s.pt")

results_hybrid = yolo_hybrid.train(
    trainer=EfficientViTYOLOTrainer,
    data="/kaggle/working/taco.yaml",

    # ── Core ──────────────────────────────────────────────────────────────
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,

    # ── Optimiser ─────────────────────────────────────────────────────────
    optimizer="AdamW",
    lr0=1e-3,
    lrf=0.01,
    weight_decay=5e-4,
    momentum=0.937,
    cos_lr=True,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,

    # ── Augmentation (same as baseline for fair comparison) ────────────────
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.4,
    close_mosaic=15,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=10.0, translate=0.1, scale=0.5, shear=2.0,
    perspective=0.0005, flipud=0.1, fliplr=0.5,

    # ── Misc ──────────────────────────────────────────────────────────────
    patience=20,
    seed=42,
    verbose=True,
    project="/kaggle/working/runs/detect",
    name="yolov8s_efficientvit"
)

print("\nHybrid training complete!")
print(f"  Best mAP50    : {results_hybrid.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"  Best mAP50-95 : {results_hybrid.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")


In [ ]:
import shutil, base64
from IPython.display import HTML

shutil.make_archive('/kaggle/working/efficientvit_100epochs', 'zip',
                    '/kaggle/working/runs/detect/yolov8s_efficientvit')

with open('/kaggle/working/efficientvit_100epochs.zip', 'rb') as f:
    data = base64.b64encode(f.read()).decode()

HTML(f'<a href="data:application/zip;base64,{data}" download="efficientvit_100epochs.zip">⬇️ Click here to download hybrid results</a>')